# Initializing

In [2]:
import os
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

In [3]:
from pathlib import Path

In [4]:
from data_platform.core.engine.spark import start_spark
from data_platform.core.utils.filesystem import read_configs
from data_platform.sources.qbo.utils.task_schedular import create_jobs

In [5]:
companies = read_configs(source_system="qbo",config_type="contracts", name="facts.json")["company_codes"]
# scope = [2024,2025,2026] 
scope = range(2018, 2027,1)
tasks = create_jobs(companies=companies, scope=scope)
path_config = read_configs(source_system="qbo",config_type="io", name="path.json")

# Flatten - Pandas

In [6]:
from data_platform.sources.qbo.transformation.engine_gl import transform_gl_pandas 
from data_platform.sources.qbo.transformation.engine_pl import transform_pl_pandas

In [7]:
df = transform_gl_pandas(tasks=tasks,scope=scope,path_config=path_config)


Total rows processed: 678427


In [8]:
df = transform_pl_pandas(tasks=tasks,scope=scope,path_config=path_config)

Total rows processed: 221559


# Flatten - Spark

In [9]:
spark = start_spark(app_name="qbo_etl")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/06 20:24:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [10]:
from data_platform.sources.qbo.transformation.engine_pl import transform_pl_spark
from data_platform.sources.qbo.transformation.engine_gl import transform_gl_spark

In [12]:
df = transform_gl_spark(tasks=tasks, scope=scope, spark=spark, path_config=path_config)
df.count()


Discovered columns superset and composed default schema


678427

In [13]:
df = transform_pl_spark(tasks=tasks, scope=scope, spark=spark, path_config=path_config)
df.count()


Discovered columns superset and composed default schema


221559

In [14]:
gl_path = Path(path_config["root"]) / path_config["silver"]["gl"]
pl_path = Path(path_config["root"]) / path_config["silver"]["pl"]
str(gl_path/"spark")

'/home/zhe_rao/projects/Database/Silver/QBO/Fact/GeneralLedger/spark'

In [15]:
df = spark.read.parquet(str(gl_path/"spark"))
df.count()

665239

In [16]:
df2 = spark.read.parquet(str(pl_path/"spark"))
df2.count()

221559

In [17]:
spark.stop()